# 09 · Conditional Logic

- `CASE WHEN ... THEN ... ELSE ... END` — SQL's if/else
- `COALESCE` — first non-NULL value
- `NULLIF` — turn a value into NULL
- the "conditional aggregation" trick (pivoting with `CASE` inside `SUM`)

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## `CASE` — bucketing values
Label products by a price tier:

In [ ]:
%%sql
SELECT product_name, unit_price,
       CASE
           WHEN unit_price < 30  THEN 'budget'
           WHEN unit_price < 100 THEN 'mid'
           ELSE 'premium'
       END AS price_tier
FROM products
ORDER BY unit_price;

## `COALESCE` — handle NULLs
Show email, or '(none)' when missing:

In [ ]:
%%sql
SELECT first_name, COALESCE(email, '(none)') AS email FROM customers;

## `NULLIF`
Returns NULL if the two args are equal — handy to avoid divide-by-zero. Here, treat 0 stock as NULL:

In [ ]:
%%sql
SELECT product_name, NULLIF(in_stock, 0) AS stock_or_null FROM products LIMIT 5;

## Conditional aggregation (a simple pivot)
Count orders by status **as columns** in a single row using `CASE` inside `SUM`:

In [ ]:
%%sql
SELECT
    SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed,
    SUM(CASE WHEN status = 'pending'   THEN 1 ELSE 0 END) AS pending,
    SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled
FROM orders;

## Practice

**✏️ Exercise 1.** Label each employee as 'leadership' if salary >= 150000, 'senior' if >= 90000, else 'staff'.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT first_name, salary,
       CASE WHEN salary >= 150000 THEN 'leadership'
            WHEN salary >= 90000  THEN 'senior'
            ELSE 'staff' END AS band
FROM employees
ORDER BY salary DESC;

**✏️ Exercise 2.** Show each product with a column 'availability' that says 'in stock' when in_stock > 0 else 'out of stock'.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT product_name, in_stock,
       CASE WHEN in_stock > 0 THEN 'in stock' ELSE 'out of stock' END AS availability
FROM products;

### ✅ Recap
`CASE` adds branching logic; `COALESCE`/`NULLIF` tame NULLs; `CASE` inside
aggregates pivots rows into columns.

**Next:** `10_window_functions.ipynb`.